CODIGO PARA BUSCA DE MHW NO ATLANTICO SUL

passo1: montar Drive e checar/criar pastas;

passo2: config do projeto (períodos, região, paths);

passo3: libs (xarray/dask/cdsapi etc.);

passo4: setup de credenciais de dados como CDS;

passo5: funções robustas de download ERA5 SST (mensal, retomável, K→°C) para a(s) região(ões) definida(s);

passo6: pré-processar p/ médias diárias e salvar NetCDF organizado

passo7: plots de series temporais e mapas

In [1]:
# PASSO 1 — Montar Google Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# Verificação rápida
import os, pathlib, sys
DRIVE_ROOT = pathlib.Path('/content/drive/MyDrive')
if not DRIVE_ROOT.exists():
    raise OSError("MyDrive não foi encontrado após o mount. Verifique a conta do Google Drive conectada.")
print("✅ Google Drive montado em:", DRIVE_ROOT)


Mounted at /content/drive
✅ Google Drive montado em: /content/drive/MyDrive


In [2]:
# PASSO 1 — Definir caminhos do projeto
from pathlib import Path

BASE_DIR   = Path("/content/drive/MyDrive/01_ESTUDOS/05_OCEANOGRAFIA")
CODES_DIR  = BASE_DIR / "05_CODIGOS"
DATA_DIR   = BASE_DIR / "06_DADOS"
PLOTS_DIR  = BASE_DIR / "07_PLOTAGENS"

for p in [BASE_DIR, CODES_DIR, DATA_DIR, PLOTS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("✅ Estrutura garantida:")
print("  ├── BASE_DIR  :", BASE_DIR)
print("  ├── CODES_DIR :", CODES_DIR)
print("  ├── DATA_DIR  :", DATA_DIR)
print("  └── PLOTS_DIR :", PLOTS_DIR)


✅ Estrutura garantida:
  ├── BASE_DIR  : /content/drive/MyDrive/01_ESTUDOS/05_OCEANOGRAFIA
  ├── CODES_DIR : /content/drive/MyDrive/01_ESTUDOS/05_OCEANOGRAFIA/05_CODIGOS
  ├── DATA_DIR  : /content/drive/MyDrive/01_ESTUDOS/05_OCEANOGRAFIA/06_DADOS
  └── PLOTS_DIR : /content/drive/MyDrive/01_ESTUDOS/05_OCEANOGRAFIA/07_PLOTAGENS


In [3]:
# PASSO 1 — Checagens de existência + marcadores .keep
(MHW_nb := CODES_DIR / "MHW.ipynb")

# cria um .keep em cada pasta para garantir sincronização
for p in [CODES_DIR, DATA_DIR, PLOTS_DIR]:
    keep = p / ".keep"
    if not keep.exists():
        keep.write_text("")  # arquivo vazio
        # sem print para não poluir; é normal criar na primeira vez

if MHW_nb.exists():
    print("✅ Notebook encontrado:", MHW_nb)
else:
    print("⚠️  AVISO: 'MHW.ipynb' ainda não está em", CODES_DIR)
    print("    (isso não impede seguir; só certifique-se de criar/salvar o notebook lá)")


✅ Notebook encontrado: /content/drive/MyDrive/01_ESTUDOS/05_OCEANOGRAFIA/05_CODIGOS/MHW.ipynb


In [4]:
# PASSO 1 — Espaço em disco e permissões básicas
import shutil, os, stat

total, used, free = shutil.disk_usage("/content/drive")
to_gb = lambda b: round(b / (1024**3), 2)
print(f"💽 Espaço no Drive -> Total: {to_gb(total)} GB | Usado: {to_gb(used)} GB | Livre: {to_gb(free)} GB")

# garantir permissões razoáveis nas pastas do projeto (rwx para dono, rx para grupo/outros)
for p in [BASE_DIR, CODES_DIR, DATA_DIR, PLOTS_DIR]:
    try:
        os.chmod(p, stat.S_IRWXU | stat.S_IRGRP | stat.S_IXGRP | stat.S_IROTH | stat.S_IXOTH)
    except Exception as e:
        # Em alguns ambientes o chmod pode não ter efeito; não é crítico
        pass
print("✅ Checagens do passo 1 concluídas.")


💽 Espaço no Drive -> Total: 107.72 GB | Usado: 42.92 GB | Livre: 64.79 GB
✅ Checagens do passo 1 concluídas.


In [5]:
# PASSO 2 — Configuração do projeto: períodos, regiões e paths (sem libs extras)

from pathlib import Path
from dataclasses import dataclass, asdict
from datetime import datetime, timezone
import json

# ===== Paths (herda do Passo 1) =====
BASE_DIR   = Path("/content/drive/MyDrive/01_ESTUDOS/04_OCEANOGRAFIA")
CODES_DIR  = BASE_DIR / "05_CODIGOS"
DATA_DIR   = BASE_DIR / "06_DADOS"
PLOTS_DIR  = BASE_DIR / "07_PLOTAGENS"

# ===== Utilitários leves =====
def to_utc(s: str) -> str:
    """
    Normaliza string ISO para formato UTC com 'Z'.
    Aceita 'YYYY-MM-DD', 'YYYY-MM-DDTHH', '...:MM', '...:SS' e adiciona 'Z'.
    """
    s_clean = s.replace(" ", "T")
    if "T" not in s_clean:
        s_clean += "T00:00:00"
    if len(s_clean.split("T")[1]) == 2:
        s_clean += ":00:00"
    if len(s_clean.split("T")[1]) == 5:
        s_clean += ":00"
    if not s_clean.endswith("Z"):
        s_clean += "Z"
    # valida
    datetime.fromisoformat(s_clean.replace("Z", "+00:00"))
    return s_clean

def bbox_era5(north, west, south, east):
    """
    Retorna bbox no formato exigido pelo CDS/ERA5: [North, West, South, East]
    com longitudes em graus leste no intervalo [-180, 180].
    """
    for val, name in [(north,"north"), (south,"south")]:
        if not (-90 <= val <= 90):
            raise ValueError(f"{name} fora de [-90,90].")
    for val, name in [(west,"west"), (east,"east")]:
        if not (-180 <= val <= 180):
            raise ValueError(f"{name} fora de [-180,180].")
    if south >= north:
        raise ValueError("south deve ser < north.")
    return [float(north), float(west), float(south), float(east)]

@dataclass
class Region:
    name: str
    north: float
    west: float
    south: float
    east: float

    def era5_area(self):
        return bbox_era5(self.north, self.west, self.south, self.east)

@dataclass
class EventWindow:
    label: str
    start: str   # ISO UTC str
    end: str     # ISO UTC str
    peak: str | None = None

    def as_iso(self):
        return {
            "start": to_utc(self.start),
            "end":   to_utc(self.end),
            "peak":  to_utc(self.peak) if self.peak else None
        }

# ===== Regiões =====
# Região ampla do Atlântico Sul (contexto MHW): 60°S–0°, 70°W–20°E
REGION_SAO_FULL = Region(
    name="SAO_FULL",
    north=0.0,   west=-70.0,
    south=-60.0, east=20.0
)

# Caixa costeira focada em SC e adjacências oceânicas (para análises locais)
# Ajuste fácil depois, se quiser mais estreita/larga.
REGION_SC_COAST = Region(
    name="SC_COAST",
    north=-24.0,  west=-55.0,
    south=-36.0,  east=-40.0
)

# ===== Janelas dos 3 eventos do TCC =====
EVENTS = [
    EventWindow(
        label="NOV_2008",
        start="2008-11-18T00",
        end="2008-11-25T00",
        peak="2008-11-23T12"
    ),
    EventWindow(
        label="OUT_2021",
        start="2021-10-10T00",
        end="2021-10-13T00",
        peak="2021-10-12T12"
    ),
    EventWindow(
        label="NOV_2022",
        start="2022-11-30T18",
        end="2022-12-03T18",
        peak="2022-12-01T12"
    ),
]

# ===== Climatologia base (para limiar p90 do método Hobday) =====
CLIM_BASE = {
    "label": "CLIM_1991_2020",
    "start": "1991-01-01",
    "end":   "2020-12-31"
}

# ===== Config ERA5 (SST em single levels) =====
ERA5_CFG = {
    "dataset": "reanalysis-era5-single-levels",
    "variable": "sea_surface_temperature",
    "product_type": "reanalysis",
    # grid 0.25° combina resolução e volume de dados razoável
    "grid": [0.25, 0.25],
    # as áreas serão escolhidas dinamicamente (SAO_FULL/SC_COAST)
}

# ===== Pacote de configuração geral =====
PROJECT_CFG = {
    "paths": {
        "BASE_DIR":   str(BASE_DIR),
        "CODES_DIR":  str(CODES_DIR),
        "DATA_DIR":   str(DATA_DIR),
        "PLOTS_DIR":  str(PLOTS_DIR),
    },
    "regions": {
        REGION_SAO_FULL.name: {
            "area": REGION_SAO_FULL.era5_area(),
            "desc": "Atlântico Sul amplo (60S–0; 70W–20E)"
        },
        REGION_SC_COAST.name: {
            "area": REGION_SC_COAST.era5_area(),
            "desc": "Faixa costeira SC e adjacências oceânicas"
        }
    },
    "events": {ev.label: ev.as_iso() for ev in EVENTS},
    "climatology": CLIM_BASE,
    "era5": ERA5_CFG,
    "units": {
        "era5_sst_input": "K",
        "era5_sst_output": "degC"
    },
    "processing": {
        "target_frequency": "daily_mean",   # será usado no Passo 6
        "calendar": "standard",
        "chunks_hint": {"time": 240, "lat": 400, "lon": 400}  # só dica p/ dask no Passo 6
    }
}

# ===== Salvar config para reuso/reprodutibilidade =====
CFG_FILE = CODES_DIR / "mhw_config.json"
with open(CFG_FILE, "w") as f:
    json.dump(PROJECT_CFG, f, indent=2)

print("✅ Configuração criada e salva em:", CFG_FILE)
print("\nResumo — Eventos (UTC):")
for k, v in PROJECT_CFG["events"].items():
    print(f"  - {k}: {v['start']}  →  {v['end']}  (peak: {v['peak']})")

print("\nResumo — Regiões (ERA5 area [N, W, S, E]):")
for k, v in PROJECT_CFG["regions"].items():
    print(f"  - {k}: {v['area']}  | {v['desc']}")


✅ Configuração criada e salva em: /content/drive/MyDrive/01_ESTUDOS/04_OCEANOGRAFIA/05_CODIGOS/mhw_config.json

Resumo — Eventos (UTC):
  - NOV_2008: 2008-11-18T00:00:00Z  →  2008-11-25T00:00:00Z  (peak: 2008-11-23T12:00:00Z)
  - OUT_2021: 2021-10-10T00:00:00Z  →  2021-10-13T00:00:00Z  (peak: 2021-10-12T12:00:00Z)
  - NOV_2022: 2022-11-30T18:00:00Z  →  2022-12-03T18:00:00Z  (peak: 2022-12-01T12:00:00Z)

Resumo — Regiões (ERA5 area [N, W, S, E]):
  - SAO_FULL: [0.0, -70.0, -60.0, 20.0]  | Atlântico Sul amplo (60S–0; 70W–20E)
  - SC_COAST: [-24.0, -55.0, -36.0, -40.0]  | Faixa costeira SC e adjacências oceânicas


In [6]:
# PASSO 2 — Sanity check da configuração salva

import json
from pathlib import Path

CFG_FILE = Path("/content/drive/MyDrive/01_ESTUDOS/04_OCEANOGRAFIA/05_CODIGOS/mhw_config.json")
with open(CFG_FILE) as f:
    cfg = json.load(f)

assert "events" in cfg and "regions" in cfg and "era5" in cfg, "Config incompleta."
assert len(cfg["events"]) == 3, "Esperava 3 eventos no TCC."

print("✅ Config válida. Eventos listados:")
for name, ev in cfg["events"].items():
    print(f"  {name}: {ev['start']} → {ev['end']} (peak: {ev['peak']})")

print("\n✅ Regiões:")
for name, reg in cfg["regions"].items():
    print(f"  {name}: {reg['area']}  |  {reg['desc']}")


✅ Config válida. Eventos listados:
  NOV_2008: 2008-11-18T00:00:00Z → 2008-11-25T00:00:00Z (peak: 2008-11-23T12:00:00Z)
  OUT_2021: 2021-10-10T00:00:00Z → 2021-10-13T00:00:00Z (peak: 2021-10-12T12:00:00Z)
  NOV_2022: 2022-11-30T18:00:00Z → 2022-12-03T18:00:00Z (peak: 2022-12-01T12:00:00Z)

✅ Regiões:
  SAO_FULL: [0.0, -70.0, -60.0, 20.0]  |  Atlântico Sul amplo (60S–0; 70W–20E)
  SC_COAST: [-24.0, -55.0, -36.0, -40.0]  |  Faixa costeira SC e adjacências oceânicas


In [7]:
# PASSO 3 — Instalar (se faltar) e importar bibliotecas do projeto (versão corrigida)
# -----------------------------------------------------------------------------------
# Flags de opcionais (altere para True se quiser desde já)
WANT_CARTOPY = True   # mapas bonitos; pode demorar um pouco para instalar
WANT_GRIB    = False  # só ative se for baixar ERA5 em GRIB; vamos usar NetCDF

import sys, subprocess, importlib.util, os
from importlib import metadata as ilmd
from pathlib import Path

def _missing(pkgs: list[str]) -> list[str]:
    out = []
    for p in pkgs:
        name = p.split("==")[0].split(">=")[0].split("[")[0]  # nome base
        if importlib.util.find_spec(name) is None:
            out.append(p)
    return out

def _pip_install(pkgs: list[str]):
    if not pkgs:
        return
    print("📦 Instalando:", ", ".join(pkgs))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-qU"] + pkgs)

# ---- listas de bibliotecas ----
CORE = [
    "numpy>=1.24",
    "pandas>=2.0",
    "xarray>=2023.12",
    "dask[distributed]>=2023.12",
    "zarr>=2.15",
    "netCDF4>=1.6.5",
    "h5netcdf>=1.2",
    "bottleneck>=1.3",
    "cftime>=1.6",
    "scipy>=1.11",
    "matplotlib>=3.7",
    "tqdm>=4.66",
    "pooch>=1.8",
    "cdsapi>=0.6",
]

OPTIONAL = []
if WANT_CARTOPY:
    OPTIONAL += ["cartopy>=0.22", "pyproj>=3.6", "shapely>=2.0"]
if WANT_GRIB:
    OPTIONAL += ["cfgrib>=0.9.10.4", "eccodes>=1.6.1"]

# ---- atualizar pip/setuptools/wheel ----
_pip_install(_missing(["pip>=24.0", "setuptools>=68", "wheel>=0.41"]))
# ---- instalar só o que faltar ----
_pip_install(_missing(CORE))
_pip_install(_missing(OPTIONAL))

# ---- imports e ajustes ----
import warnings; warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import xarray as xr
import dask
from dask.diagnostics import ProgressBar
import zarr
import netCDF4
import h5netcdf
import cftime
import scipy
import matplotlib
import matplotlib.pyplot as plt
import pooch
import json
import tqdm as tqdm_mod  # para obter __version__ corretamente
from tqdm.auto import tqdm

# opcionais
if WANT_CARTOPY:
    import cartopy
    import cartopy.crs as ccrs
    import cartopy.feature as cfeature
if WANT_GRIB:
    import cfgrib  # noqa

# ---- preferências de desempenho/estética ----
xr.set_options(keep_attrs=True, display_style="text")
dask.config.set({"array.slicing.split_large_chunks": True})
ProgressBar().register()

plt.rcParams.update({
    "figure.figsize": (9, 5),
    "figure.dpi": 110,
    "savefig.bbox": "tight",
})

# ---- util para pegar versões de forma robusta ----
def _ver_pkg(pkg: str, fallback_module=None):
    try:
        return ilmd.version(pkg)
    except ilmd.PackageNotFoundError:
        if fallback_module is not None:
            try:
                return fallback_module.__version__
            except Exception:
                return "?"
        return "?"

versions = {
    "numpy":       _ver_pkg("numpy", np),
    "pandas":      _ver_pkg("pandas", pd),
    "xarray":      _ver_pkg("xarray", xr),
    "dask":        _ver_pkg("dask", dask),
    "zarr":        _ver_pkg("zarr", zarr),
    "netCDF4":     _ver_pkg("netCDF4", netCDF4),
    "h5netcdf":    _ver_pkg("h5netcdf", h5netcdf),
    "cftime":      _ver_pkg("cftime", cftime),
    "scipy":       _ver_pkg("scipy", scipy),
    "matplotlib":  _ver_pkg("matplotlib", matplotlib),
    "tqdm":        _ver_pkg("tqdm", tqdm_mod),
    "pooch":       _ver_pkg("pooch", pooch),
    "cdsapi":      _ver_pkg("cdsapi"),
}
if WANT_CARTOPY:
    versions["cartopy"] = _ver_pkg("cartopy", cartopy)
if WANT_GRIB:
    versions["cfgrib"]  = _ver_pkg("cfgrib")
    versions["eccodes"] = _ver_pkg("eccodes")

print("✅ Bibliotecas carregadas.\n" + "-"*60)
for k, v in versions.items():
    print(f"{k:>12s}: {v}")
print("-"*60)

# sanity check de escrita em diretórios do projeto
BASE_DIR   = Path("/content/drive/MyDrive/01_ESTUDOS/04_OCEANOGRAFIA")
for sub in ("05_CODIGOS", "06_DADOS", "07_PLOTAGENS"):
    p = BASE_DIR / sub
    p.mkdir(parents=True, exist_ok=True)
    test = p / ".write_test"
    try:
        test.write_text("ok")
        test.unlink(missing_ok=True)
    except Exception as e:
        print(f"⚠️  Sem permissão de escrita em {p}: {e}")
print("🧩 Passo 3 concluído.")


✅ Bibliotecas carregadas.
------------------------------------------------------------
       numpy: 2.0.2
      pandas: 2.2.2
      xarray: 2025.9.0
        dask: 2025.5.0
        zarr: 3.1.3
     netCDF4: 1.7.2
    h5netcdf: 1.6.4
      cftime: 1.6.4.post1
       scipy: 1.16.2
  matplotlib: 3.10.0
        tqdm: 4.67.1
       pooch: 1.8.2
      cdsapi: 0.7.7
     cartopy: 0.25.0
------------------------------------------------------------
🧩 Passo 3 concluído.


In [8]:
# PASSO 4 — CDS: gravar ~/.cdsapirc com entrada mascarada (Colab)
from getpass import getpass
import pathlib, textwrap

print("Cole seu UID (número) e API key do CDS (perfil > 'API key').")
uid = input("CDS UID: ").strip()
api = getpass("CDS API key (oculta): ").strip()

cfg = textwrap.dedent(f"""\
    url: https://cds.climate.copernicus.eu/api/v2
    key: {uid}:{api}
    verify: 1
""")

path = pathlib.Path.home() / ".cdsapirc"
path.write_text(cfg)
print(f"✅ .cdsapirc gravado em {path}")
# teste rápido do cliente (sem solicitar dados)
import cdsapi
c = cdsapi.Client()
print("✅ cdsapi pronto:", c.url)


Cole seu UID (número) e API key do CDS (perfil > 'API key').
CDS UID: 03ac90e2-6ad0-4cce-8c24-1e93f7a1b361
CDS API key (oculta): ··········
✅ .cdsapirc gravado em /root/.cdsapirc
✅ cdsapi pronto: https://cds.climate.copernicus.eu/api/v2


In [9]:
# ===========================
# PASSO 5 — DOWNLOAD HELPERS (ROBUSTO E LEVE)
# ERA5 (SST horária via CDS) + OISST v2.1 (SST diária via NCEI)
# ===========================
import os, json, shutil, time, datetime as dt
from pathlib import Path
from typing import Dict, List, Tuple
import pandas as pd
import xarray as xr
import requests

# ---- paths e config
DRIVE_ROOT = "/content/drive/MyDrive/01_ESTUDOS/04_OCEANOGRAFIA"
DIR_DADOS  = f"{DRIVE_ROOT}/06_DADOS"
CFG_PATH   = f"{DRIVE_ROOT}/05_CODIGOS/mhw_config.json"

assert os.path.exists(DRIVE_ROOT), f"Drive root não montado: {DRIVE_ROOT}"
assert os.path.exists(CFG_PATH), f"Config não encontrada: {CFG_PATH}"

with open(CFG_PATH, "r") as f:
    CFG = json.load(f)

EVENTS: Dict[str, Dict] = CFG["events"]
REGIONS_RAW: Dict[str, Dict] = CFG["regions"]

# pastas de saída
ERA5_RAW_ROOT  = f"{DIR_DADOS}/ERA5/SST/raw"
OISST_RAW_ROOT = f"{DIR_DADOS}/OISST/SST/raw"
Path(ERA5_RAW_ROOT).mkdir(parents=True, exist_ok=True)
Path(OISST_RAW_ROOT).mkdir(parents=True, exist_ok=True)

# ===========================
# Regiões (normalização)
# ===========================
def normalize_region(region_entry: Dict) -> Dict[str, float]:
    if "area" in region_entry:
        a = region_entry["area"]
        assert isinstance(a, (list, tuple)) and len(a) == 4, f"Formatação inválida de 'area': {a}"
        N, W, S, E = a
        return {"north": float(N), "west": float(W), "south": float(S), "east": float(E)}
    keys = {k.lower(): v for k, v in region_entry.items()}
    for need in ("north", "west", "south", "east"):
        assert need in keys, f"Chave ausente em região: {need}"
    return {k: float(keys[k]) for k in ("north", "west", "south", "east")}

REGIONS: Dict[str, Dict[str, float]] = {name: normalize_region(r) for name, r in REGIONS_RAW.items()}

print("Regiões interpretadas:")
for k, r in REGIONS.items():
    print(f"  - {k}: {r}")

# ===========================
# Tempo/meses
# ===========================
def month_span(start: str, end: str) -> List[Tuple[int,int]]:
    # Remove timezone para evitar warnings
    t0 = pd.to_datetime(start, utc=True).tz_convert(None)
    t1 = pd.to_datetime(end,   utc=True).tz_convert(None)
    months = pd.period_range(t0.to_period("M"), t1.to_period("M"), freq="M")
    return [(p.year, p.month) for p in months]

def day_strings_in_month(year: int, month: int) -> List[str]:
    first = dt.date(year, month, 1)
    nextm = dt.date(year+1, 1, 1) if month == 12 else dt.date(year, month+1, 1)
    ndays = (nextm - first).days
    return [f"{d:02d}" for d in range(1, ndays+1)]

def calendar_days(year: int, month: int) -> List[dt.date]:
    first = dt.date(year, month, 1)
    nextm = dt.date(year+1, 1, 1) if month == 12 else dt.date(year, month+1, 1)
    ndays = (nextm - first).days
    return [first + dt.timedelta(days=i) for i in range(ndays)]

def list_event_months(events: Dict[str, Dict]) -> List[Tuple[str,int,int]]:
    rows = []
    for name, e in events.items():
        for (y, m) in month_span(e["start"], e["end"]):
            rows.append((name, y, m))
    return sorted(set(rows))

EVENT_MONTHS = list_event_months(EVENTS)

def months_by_event() -> Dict[str, List[Tuple[int,int]]]:
    out = {}
    for ev, meta in EVENTS.items():
        out[ev] = month_span(meta["start"], meta["end"])
    return out

def pretty_event_table():
    mb = months_by_event()
    print("\nMeses por evento:")
    for ev, rows in mb.items():
        labs = [f"{y}-{m:02d}" for (y, m) in rows]
        print(f"  - {ev}: {', '.join(labs)}")

pretty_event_table()

# ===========================
# ERA5 via CDSAPI
# ===========================
def ensure_cdsapi():
    try:
        import cdsapi  # noqa
    except Exception as e:
        raise RuntimeError("cdsapi não está instalado (ver Passo 3).") from e

def era5_bbox(region: Dict[str, float]) -> List[float]:
    return [region["north"], region["west"], region["south"], region["east"]]

def download_era5_sst_month(region_name: str, year: int, month: int,
                            retries: int = 3, sleep_s: int = 15) -> str:
    ensure_cdsapi()
    import cdsapi
    region = REGIONS[region_name]
    area   = era5_bbox(region)
    out_dir = f"{ERA5_RAW_ROOT}/{region_name}/{year}"
    Path(out_dir).mkdir(parents=True, exist_ok=True)
    yyyymm = f"{year}{month:02d}"
    target = f"{out_dir}/era5_sst_{region_name}_{yyyymm}.nc"
    if os.path.exists(target) and os.path.getsize(target) > 0:
        print(f"[ERA5] OK (skip): {target}")
        return target
    req = {
        "product_type": "reanalysis",
        "variable": ["sea_surface_temperature"],
        "year":   str(year),
        "month":  f"{month:02d}",
        "day":    day_strings_in_month(year, month),
        "time":   [f"{h:02d}:00" for h in range(24)],
        "area":   area,
        "format": "netcdf",
    }
    tmp = target + ".part"
    last_err = None
    for k in range(1, retries+1):
        try:
            print(f"[ERA5] requisitando {region_name} {yyyymm} (tentativa {k}/{retries}) …")
            c = cdsapi.Client()
            c.retrieve("reanalysis-era5-single-levels", req, tmp)
            shutil.move(tmp, target)
            print(f"[ERA5] salvo: {target}")
            return target
        except Exception as e:
            last_err = e
            print(f"[ERA5] falhou tentativa {k}: {e}")
            time.sleep(sleep_s * k)
    raise RuntimeError(f"[ERA5] falhou baixar {region_name} {yyyymm}") from last_err

# ===========================
# OISST v2.1 — diário via NCEI (leve e à prova de crash)
# ===========================
NCEI_BASE = "https://www.ncei.noaa.gov/data/sea-surface-temperature-optimum-interpolation/v2.1/access/avhrr"
REQ_HEADERS = {"User-Agent": "oisst-downloader/1.0"}
OISST_CHUNKS = {"time": 5, "lat": 200, "lon": 200}  # chunks modestos p/ memória estável

def oisst_daily_url(d: dt.date) -> str:
    ym = f"{d.year}{d.month:02d}"
    ymd = f"{d.year}{d.month:02d}{d.day:02d}"
    return f"{NCEI_BASE}/{ym}/oisst-avhrr-v02r01.{ymd}.nc"

def _snap(val: float, step: float, base: float, vmin: float, vmax: float) -> float:
    n = round((val - base) / step)
    s = base + n * step
    return min(max(s, vmin), vmax)

def _region_bounds_for_oisst(region: Dict[str, float]) -> Tuple[float,float,float,float,bool]:
    lat_min = min(region["south"], region["north"])
    lat_max = max(region["south"], region["north"])
    lat_min = _snap(lat_min, 0.25, -89.875, -89.875,  89.875)
    lat_max = _snap(lat_max, 0.25, -89.875, -89.875,  89.875)
    def to360(lon):
        x = lon % 360.0
        return x if x >= 0 else x + 360.0
    w360 = _snap(to360(region["west"]), 0.25, 0.125, 0.125, 359.875)
    e360 = _snap(to360(region["east"]), 0.25, 0.125, 0.125, 359.875)
    wrap = w360 > e360
    return lat_min, lat_max, w360, e360, wrap

def _download_file(url: str, out_path: Path, retries: int = 3, sleep_s: int = 10, timeout: int = 600):
    if out_path.exists() and out_path.stat().st_size > 0:
        return
    tmp = Path(str(out_path) + ".part")
    last_err = None
    for k in range(1, retries+1):
        try:
            print(f"[OISST] baixando {url} (tentativa {k}/{retries}) …")
            with requests.get(url, stream=True, timeout=timeout, headers=REQ_HEADERS) as r:
                r.raise_for_status()
                with open(tmp, "wb") as f:
                    for chunk in r.iter_content(chunk_size=1024*1024):
                        if chunk:
                            f.write(chunk)
            tmp.replace(out_path)
            return
        except Exception as e:
            last_err = e
            print(f"[OISST] falhou download ({k}/{retries}): {e}")
            time.sleep(sleep_s * k)
    raise RuntimeError(f"Falha ao baixar: {url}") from last_err

def _validate_oisst_file(nc_path: Path) -> bool:
    try:
        ds = xr.open_dataset(nc_path, engine="netcdf4")
        ok = ("sst" in ds.variables) and all(dim in ds.dims for dim in ("time","lat","lon"))
        ds.close()
        return bool(ok)
    except Exception:
        return False

def _ensure_daily_ok(date: dt.date, folder: Path) -> Path:
    url = oisst_daily_url(date)
    out = folder / Path(url).name
    _download_file(url, out)
    if not _validate_oisst_file(out):
        try: out.unlink(missing_ok=True)
        except Exception: pass
        _download_file(url, out)
        if not _validate_oisst_file(out):
            raise RuntimeError(f"Arquivo inválido após re-download: {out}")
    return out

def _preproc_subset(lat_min, lat_max, lon_slice):
    # Devolve função preprocess para open_mfdataset, recortando e mantendo só 'sst'
    def _pp(ds):
        _ds = ds[["sst"]]
        _ds = _ds.sel(lat=slice(lat_min, lat_max))
        _ds = _ds.sel(lon=lon_slice)
        return _ds
    return _pp

def _open_month_subset(daily_files: List[str],
                       lat_min: float, lat_max: float,
                       w360: float, e360: float, wrap: bool) -> xr.Dataset:
    # Abre já recortado para economizar memória. Trata wrap em 0–360.
    if not wrap:
        pre = _preproc_subset(lat_min, lat_max, slice(w360, e360))
        ds = xr.open_mfdataset(
            daily_files,
            combine="by_coords",
            preprocess=pre,
            parallel=False,
            data_vars="minimal",
            coords="minimal",
            compat="override",
            engine="netcdf4",
            chunks=OISST_CHUNKS,
        )
        return ds[["sst"]]
    else:
        pre_left  = _preproc_subset(lat_min, lat_max, slice(w360, 359.875))
        pre_right = _preproc_subset(lat_min, lat_max, slice(0.125, e360))
        ds_left = xr.open_mfdataset(
            daily_files, combine="by_coords", preprocess=pre_left,
            parallel=False, data_vars="minimal", coords="minimal",
            compat="override", engine="netcdf4", chunks=OISST_CHUNKS,
        )
        ds_right = xr.open_mfdataset(
            daily_files, combine="by_coords", preprocess=pre_right,
            parallel=False, data_vars="minimal", coords="minimal",
            compat="override", engine="netcdf4", chunks=OISST_CHUNKS,
        )
        ds = xr.concat([ds_left[["sst"]], ds_right[["sst"]]], dim="lon")
        return ds

def download_oisst_month(region_name: str, year:int, month:int,
                         keep_daily_tmp: bool=False) -> str:
    """
    Baixa OISST diário (global), valida/repara, abre cada arquivo já recortando
    por região (preprocess) e concatena por tempo com baixo uso de RAM.
    Saída: OISST/SST/raw/{region}/{year}/oisst_sst_{region}_{YYYYMM}.nc
    """
    region = REGIONS[region_name]
    out_dir = Path(f"{OISST_RAW_ROOT}/{region_name}/{year}")
    out_dir.mkdir(parents=True, exist_ok=True)
    yyyymm = f"{year}{month:02d}"
    target = out_dir / f"oisst_sst_{region_name}_{yyyymm}.nc"
    if target.exists() and target.stat().st_size > 0:
        print(f"[OISST] OK (skip): {target}")
        return str(target)

    # diretório temporário de diários
    tmp_dir = out_dir / f".tmp_{yyyymm}"
    tmp_dir.mkdir(parents=True, exist_ok=True)

    # 1) garantir todos os dias íntegros
    daily_files = []
    for d in calendar_days(year, month):
        try:
            p = _ensure_daily_ok(d, tmp_dir)
            daily_files.append(str(p))
        except Exception as e:
            print(f"[OISST] aviso: {d} ignorado ({e})")

    if not daily_files:
        raise RuntimeError(f"[OISST] nenhum arquivo diário válido para {yyyymm} em {region_name}")

    # 2) abrir já recortado (memória baixa)
    lat_min, lat_max, w360, e360, wrap = _region_bounds_for_oisst(region)
    ds = _open_month_subset(daily_files, lat_min, lat_max, w360, e360, wrap)

    # 3) unidade em °C (normalmente já está)
    units = str(ds["sst"].attrs.get("units", "")).lower()
    if units in ("k", "kelvin", "degree_k", "degrees_k"):
        ds["sst"] = ds["sst"] - 273.15
        ds["sst"].attrs["units"] = "degree_Celsius"

    # 4) escrita final
    tmp_out = Path(str(target) + ".part")
    encoding = {"sst": {"zlib": True, "complevel": 4, "dtype": "float32", "chunksizes": (1, 200, 200)}}
    ds.to_netcdf(tmp_out, encoding=encoding)
    ds.close()
    tmp_out.replace(target)
    print(f"[OISST] salvo: {target}")

    # 5) limpeza
    if not keep_daily_tmp:
        for p in tmp_dir.glob("*.nc"):
            try: p.unlink()
            except Exception: pass
        try: tmp_dir.rmdir()
        except OSError: pass

    return str(target)

# ===========================
# DRIVER — reexecutável (com skip)
# ===========================
print("\nDriver: eventos/meses =", EVENT_MONTHS)
print("Driver: regiões       =", list(REGIONS.keys()))

# ERA5
for region_name in REGIONS.keys():
    for (_ev, y, m) in EVENT_MONTHS:
        download_era5_sst_month(region_name, y, m)

# OISST (leve e robusto)
for region_name in REGIONS.keys():
    for (_ev, y, m) in EVENT_MONTHS:
        download_oisst_month(region_name, y, m)

print("\n✅ Passo 5 concluído (ERA5+OISST brutos prontos).")


Regiões interpretadas:
  - SAO_FULL: {'north': 0.0, 'west': -70.0, 'south': -60.0, 'east': 20.0}
  - SC_COAST: {'north': -24.0, 'west': -55.0, 'south': -36.0, 'east': -40.0}

Meses por evento:
  - NOV_2008: 2008-11
  - OUT_2021: 2021-10
  - NOV_2022: 2022-11, 2022-12

Driver: eventos/meses = [('NOV_2008', 2008, 11), ('NOV_2022', 2022, 11), ('NOV_2022', 2022, 12), ('OUT_2021', 2021, 10)]
Driver: regiões       = ['SAO_FULL', 'SC_COAST']
[ERA5] requisitando SAO_FULL 200811 (tentativa 1/3) …


2025-10-01 13:00:18,431 INFO Sending request to https://cds.climate.copernicus.eu/api/v2/resources/reanalysis-era5-single-levels
INFO:cdsapi:Sending request to https://cds.climate.copernicus.eu/api/v2/resources/reanalysis-era5-single-levels


[ERA5] falhou tentativa 1: Exception: Climate Data Store API endpoint not found

It seems you are trying to access an API endpoint that does not exist.

It is possible that you are pointing to an outdated API endpoint which has been removed/renamed. 

Your api client version and/or configuration file may need to be updated.

For users of the standard cdsapi Python library, please proceed as follows:
- update your cdsapi client to the latest version available
- update your .cdsapirc file as expected ("url" should be set to https://cds.climate.copernicus.eu/api and "key" should be set to your API key WITHOUT the deprecated `<UID>:` prefix)

For users of the ecmwf-datastores-client Python library (recommended for advanced users), please ensure that your ecmwf-datastores-client version is updated and your .ecmwfdatastoresrc file is correctly configured.

If you are trying to access a REST API endpoint directly, please refer to the documentation for more information on how to access REST AP

2025-10-01 13:00:33,700 INFO Sending request to https://cds.climate.copernicus.eu/api/v2/resources/reanalysis-era5-single-levels
INFO:cdsapi:Sending request to https://cds.climate.copernicus.eu/api/v2/resources/reanalysis-era5-single-levels


[ERA5] requisitando SAO_FULL 200811 (tentativa 2/3) …
[ERA5] falhou tentativa 2: Exception: Climate Data Store API endpoint not found

It seems you are trying to access an API endpoint that does not exist.

It is possible that you are pointing to an outdated API endpoint which has been removed/renamed. 

Your api client version and/or configuration file may need to be updated.

For users of the standard cdsapi Python library, please proceed as follows:
- update your cdsapi client to the latest version available
- update your .cdsapirc file as expected ("url" should be set to https://cds.climate.copernicus.eu/api and "key" should be set to your API key WITHOUT the deprecated `<UID>:` prefix)

For users of the ecmwf-datastores-client Python library (recommended for advanced users), please ensure that your ecmwf-datastores-client version is updated and your .ecmwfdatastoresrc file is correctly configured.

If you are trying to access a REST API endpoint directly, please refer to the docum

KeyboardInterrupt: 

In [10]:
# ===========================
# PASSO 6 — TESTE ROBUSTO DOS BRUTOS (ERA5 + OISST)
# Verifica variáveis, dimensões, ranges e amostras; imprime diagnósticos úteis.
# ===========================
import os, json
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr
import dask
dask.config.set(scheduler="threads")

# --- paths e config
DRIVE_ROOT = "/content/drive/MyDrive/01_ESTUDOS/04_OCEANOGRAFIA"
DIR_DADOS  = f"{DRIVE_ROOT}/06_DADOS"
CFG_PATH   = f"{DRIVE_ROOT}/05_CODIGOS/mhw_config.json"

with open(CFG_PATH, "r") as f:
    CFG = json.load(f)

EVENTS   = CFG["events"]
REGIONS0 = CFG["regions"]
REGIONS  = {k: (v["area"] if "area" in v else [v["north"], v["west"], v["south"], v["east"]]) for k,v in REGIONS0.items()}
REGIONS  = {k: dict(north=a[0], west=a[1], south=a[2], east=a[3]) for k,a in REGIONS.items()}

ERA5_RAW_ROOT  = f"{DIR_DADOS}/ERA5/SST/raw"
OISST_RAW_ROOT = f"{DIR_DADOS}/OISST/SST/raw"

def month_span(start, end):
    t0 = pd.to_datetime(start); t1 = pd.to_datetime(end)
    per = pd.period_range(t0.to_period("M"), t1.to_period("M"), freq="M")
    return [(p.year, p.month) for p in per]

EVENT_MONTHS = sorted({(name, y, m) for name,e in EVENTS.items() for (y,m) in month_span(e["start"], e["end"])})

# --- utilidades de descoberta robusta
def find_dim_name(ds, candidates):
    # 1) procurar entre dims
    for c in candidates:
        if c in ds.dims: return c
    # 2) procurar entre coords
    for c in candidates:
        if c in ds.coords: return c
    # 3) procurar por variáveis 1-D com attrs CF
    for name, da in ds.variables.items():
        if getattr(da, "ndim", 0) != 1:
            continue
        u = str(da.attrs.get("units", "")).lower()
        sn = str(da.attrs.get("standard_name", "")).lower()
        ln = str(da.attrs.get("long_name", "")).lower()
        if any(k in candidates for k in [name.lower()]):
            return name
        if any(k in sn for k in ["latitude","longitude","time"]):
            return name
        if "degree" in u and ("east" in u or "north" in u):
            return name
    return None

def discover_coords(ds):
    lat = find_dim_name(ds, ["latitude","lat","y"])
    lon = find_dim_name(ds, ["longitude","lon","x"])
    tim = find_dim_name(ds, ["time","valid_time","t"])
    return lat, lon, tim

def find_sst_var(ds):
    # 1) candidatos diretos
    for k in ["sst", "sea_surface_temperature", "SST", "SSTK", "analysis_sst"]:
        if k in ds.data_vars: return k
        if k in ds.variables: return k
    # 2) por atributos CF
    for name, da in ds.data_vars.items():
        std = str(da.attrs.get("standard_name","")).lower()
        lng = str(da.attrs.get("long_name","")).lower()
        if "sea_surface_temperature" in std or "sea surface temperature" in lng:
            return name
    # 3) fallback: variável 3-D com dims contendo tempo/lat/lon
    lat, lon, tim = discover_coords(ds)
    if lat and lon and tim:
        for name, da in ds.data_vars.items():
            if all(d in da.dims for d in [tim, lat, lon]):
                return name
    return None

def nice_rng(values):
    vmin = float(np.nanmin(values))
    vmax = float(np.nanmax(values))
    return f"{vmin:.3f} .. {vmax:.3f}"

def quick_sample_stats(ds, var, lat, lon, tim):
    # Amostra leve (subamostragem espacial + primeiros/últimos tempos)
    sl_lat = slice(0, None, max(1, ds.dims[lat]//100 or 1))
    sl_lon = slice(0, None, max(1, ds.dims[lon]//100 or 1))
    a0 = ds[var].isel({tim: 0, lat: sl_lat, lon: sl_lon})
    a1 = ds[var].isel({tim:-1, lat: sl_lat, lon: sl_lon})
    v0 = float(a0.min().compute()); V0 = float(a0.max().compute())
    v1 = float(a1.min().compute()); V1 = float(a1.max().compute())
    return v0, V0, v1, V1

def exist_ok(p):
    return os.path.exists(p) and os.path.getsize(p) > 0

# --- paths
def era5_path(region, y, m):  return f"{ERA5_RAW_ROOT}/{region}/{y}/era5_sst_{region}_{y}{m:02d}.nc"
def oisst_path(region, y, m): return f"{OISST_RAW_ROOT}/{region}/{y}/oisst_sst_{region}_{y}{m:02d}.nc"

print("=== TESTE ROBUSTO ===")
print("Regiões:", {k: v for k,v in REGIONS.items()})
print("Eventos/meses:", EVENT_MONTHS, "\n")

ok_era5 = ok_oisst = 0
diag_era5 = []

# --- loop
for region_name in REGIONS.keys():
    for (_ev, y, m) in EVENT_MONTHS:
        # ERA5
        f = era5_path(region_name, y, m)
        if exist_ok(f):
            try:
                ds = xr.open_dataset(f, chunks={"time": 24}, engine="netcdf4")
                lat, lon, tim = discover_coords(ds)
                var = find_sst_var(ds)
                if not (lat and lon and tim and var):
                    diag = {
                        "file": f, "dims": dict(ds.dims),
                        "coords": list(ds.coords),
                        "vars": list(ds.data_vars)[:6],
                        "guess_lat": lat, "guess_lon": lon, "guess_time": tim, "guess_var": var
                    }
                    diag_era5.append(diag)
                    print(f"[ERA5][ERRO] {region_name} {y}-{m:02d} → variáveis/dimensões não localizadas.")
                else:
                    units = str(ds[var].attrs.get("units","")).lower()
                    # Heurística simples K→°C
                    sample = float(ds[var].isel({tim:0, lat:0, lon:0}).values)
                    is_kelvin = ("k" == units or units.startswith("kelvin")) or (sample > 100)
                    v0,V0,v1,V1 = quick_sample_stats(ds, var, lat, lon, tim)
                    if is_kelvin:
                        v0-=273.15; V0-=273.15; v1-=273.15; V1-=273.15
                        udisp = "°C (de K)"
                    else:
                        udisp = ds[var].attrs.get("units","")
                    t0 = str(pd.to_datetime(ds[tim].values[0]))
                    t1 = str(pd.to_datetime(ds[tim].values[-1]))
                    print(f"[ERA5][OK]  {region_name} {y}-{m:02d} "
                          f"shape=({ds.dims[tim]},{ds.dims[lat]},{ds.dims[lon]}), "
                          f"time={t0} → {t1}, "
                          f"lat={nice_rng(ds[lat].values)}, lon={nice_rng(ds[lon].values)}, "
                          f"SST@{t0[:10]} [{v0:.2f},{V0:.2f}] {udisp}; "
                          f"SST@{t1[:10]} [{v1:.2f},{V1:.2f}]")
                    ok_era5 += 1
                ds.close()
            except Exception as e:
                print(f"[ERA5][ERRO] {region_name} {y}-{m:02d}: {e}")
        else:
            print(f"[ERA5][MISS] {f}")

        # OISST
        g = oisst_path(region_name, y, m)
        if exist_ok(g):
            try:
                ds = xr.open_dataset(g, chunks={"time": 31}, engine="netcdf4")
                lat, lon, tim = discover_coords(ds)
                var = find_sst_var(ds) or "sst"
                v0,V0,v1,V1 = quick_sample_stats(ds, var, lat, lon, tim)
                t0 = str(pd.to_datetime(ds[tim].values[0]))
                t1 = str(pd.to_datetime(ds[tim].values[-1]))
                units = ds[var].attrs.get("units","°C")
                print(f"[OISST][OK] {region_name} {y}-{m:02d} "
                      f"shape=({ds.dims[tim]},{ds.dims[lat]},{ds.dims[lon]}), "
                      f"time={t0} → {t1}, "
                      f"lat={nice_rng(ds[lat].values)}, lon={nice_rng(ds[lon].values)}, "
                      f"SST@{t0[:10]} [{v0:.2f},{V0:.2f}] {units}; "
                      f"SST@{t1[:10]} [{v1:.2f},{V1:.2f}]")
                ok_oisst += 1
                ds.close()
            except Exception as e:
                print(f"[OISST][ERRO] {region_name} {y}-{m:02d}: {e}")
        else:
            print(f"[OISST][MISS] {g}")

print("\nResumo:")
print(f"  ERA5  OK: {ok_era5}")
print(f"  OISST OK: {ok_oisst}")

# Diagnóstico extra quando ERA5 falhar (mostra estrutura do arquivo para ajuste fino)
if diag_era5:
    print("\nDiagnóstico ERA5 (primeiros 2 itens):")
    for d in diag_era5[:2]:
        print(" - arquivo:", d["file"])
        print("   dims  :", d["dims"])
        print("   coords:", d["coords"])
        print("   vars  :", d["vars"])
        print("   guesses -> lat:", d["guess_lat"], "lon:", d["guess_lon"], "time:", d["guess_time"], "var:", d["guess_var"])


=== TESTE ROBUSTO ===
Regiões: {'SAO_FULL': {'north': 0.0, 'west': -70.0, 'south': -60.0, 'east': 20.0}, 'SC_COAST': {'north': -24.0, 'west': -55.0, 'south': -36.0, 'east': -40.0}}
Eventos/meses: [('NOV_2008', 2008, 11), ('NOV_2022', 2022, 11), ('NOV_2022', 2022, 12), ('OUT_2021', 2021, 10)] 

[ERA5][MISS] /content/drive/MyDrive/01_ESTUDOS/04_OCEANOGRAFIA/06_DADOS/ERA5/SST/raw/SAO_FULL/2008/era5_sst_SAO_FULL_200811.nc
[OISST][MISS] /content/drive/MyDrive/01_ESTUDOS/04_OCEANOGRAFIA/06_DADOS/OISST/SST/raw/SAO_FULL/2008/oisst_sst_SAO_FULL_200811.nc
[ERA5][MISS] /content/drive/MyDrive/01_ESTUDOS/04_OCEANOGRAFIA/06_DADOS/ERA5/SST/raw/SAO_FULL/2022/era5_sst_SAO_FULL_202211.nc
[OISST][MISS] /content/drive/MyDrive/01_ESTUDOS/04_OCEANOGRAFIA/06_DADOS/OISST/SST/raw/SAO_FULL/2022/oisst_sst_SAO_FULL_202211.nc
[ERA5][MISS] /content/drive/MyDrive/01_ESTUDOS/04_OCEANOGRAFIA/06_DADOS/ERA5/SST/raw/SAO_FULL/2022/era5_sst_SAO_FULL_202212.nc
[OISST][MISS] /content/drive/MyDrive/01_ESTUDOS/04_OCEANOGRAFIA/

In [11]:
# ===========================
# PASSO 7A (rev) — MOSAICO DE MAPAS DIÁRIOS (MHW) COM RÓTULOS °C E COLORBAR DEDICADO
# - Mantém boas práticas: paleta cmocean.thermal, interpolação bilinear, isolinhas
# - Adiciona rótulos das isolinhas em °C
# - Usa GridSpec com uma COLUNA EXCLUSIVA para a barra de cores (não sobrepõe painéis)
# - Salva em .../07_PLOTAGENS/{PRODUCT}/{REGION}/
# ===========================
import os, sys, json, subprocess
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

# --------- caminhos / config do projeto ---------
DRIVE_ROOT = "/content/drive/MyDrive/01_ESTUDOS/04_OCEANOGRAFIA"
DIR_DADOS  = f"{DRIVE_ROOT}/06_DADOS"
CFG_PATH   = f"{DRIVE_ROOT}/05_CODIGOS/mhw_config.json"
ERA5_RAW_ROOT  = f"{DIR_DADOS}/ERA5/SST/raw"
OISST_RAW_ROOT = f"{DIR_DADOS}/OISST/SST/raw"
PLOTS_ROOT     = f"{DRIVE_ROOT}/07_PLOTAGENS"
Path(PLOTS_ROOT).mkdir(parents=True, exist_ok=True)

# --------- parâmetros do mosaico ---------
PRODUCT          = "OISST"            # "OISST" (recomendado p/ mapas) ou "ERA5"
REGION_FOR_DATA  = "SAO_FULL"         # área ampla para contexto
EVENTS_TO_PLOT   = ["NOV_2008", "OUT_2021", "NOV_2022"]
UPSAMPLE_FACTOR  = 3                  # suavização espacial
ISO_DX           = 1.0                # passo das isolinhas (°C)
SHOW_ISO_LABELS  = True               # rótulos “XX°C” nas isolinhas
SAVE_DPI         = 180
FIGSIZE_BASE     = (12, 8)            # será escalado conforme nº de painéis

# ===========================
# Utilidades
# ===========================
def _exist_ok(p): return os.path.exists(p) and os.path.getsize(p) > 0
def _era5_path(region, y, m):  return f"{ERA5_RAW_ROOT}/{region}/{y}/era5_sst_{region}_{y}{m:02d}.nc"
def _oisst_path(region, y, m): return f"{OISST_RAW_ROOT}/{region}/{y}/oisst_sst_{region}_{y}{m:02d}.nc"
def _as_naive_utc(ts): return pd.to_datetime(ts, utc=True).tz_localize(None)

def _open_cfg(path):
    with open(path, "r") as f:
        cfg = json.load(f)
    regs = {}
    for k, v in cfg["regions"].items():
        if "area" in v:
            N, W, S, E = v["area"]
            regs[k] = dict(north=float(N), west=float(W), south=float(S), east=float(E))
        else:
            vv = {kk.lower(): vv for kk, vv in v.items()}
            regs[k] = dict(north=float(vv["north"]), west=float(vv["west"]),
                           south=float(vv["south"]), east=float(vv["east"]))
    return cfg, regs

def _event_months(e):
    t0 = pd.to_datetime(e["start"]); t1 = pd.to_datetime(e["end"])
    per = pd.period_range(t0.to_period("M"), t1.to_period("M"), freq="M")
    return [(p.year, p.month) for p in per]

def _discover_coords(ds):
    def pick(cands):
        for c in cands:
            if c in ds.coords or c in ds.dims: return c
        for name, da in ds.variables.items():
            if getattr(da, "ndim", 0) == 1:
                std = str(da.attrs.get("standard_name","")).lower()
                if name in cands: return name
                if any(c.startswith("lat") for c in cands) and "latitude" in std: return name
                if any(c.startswith("lon") for c in cands) and "longitude" in std: return name
                if any(c.startswith("time") for c in cands) and "time" in std: return name
        return None
    lat = pick(["latitude","lat","y"])
    lon = pick(["longitude","lon","x"])
    tim = pick(["time","valid_time","t"])
    if not all([lat,lon,tim]):
        raise ValueError(f"Coords não encontradas. dims={list(ds.dims)}, coords={list(ds.coords)}")
    return lat, lon, tim

def _find_sst_var(ds):
    for k in ["sea_surface_temperature","sst","SST","sstk","SSTK","analysis_sst"]:
        if k in ds.data_vars or k in ds.variables: return k
    lat, lon, tim = _discover_coords(ds)
    for name, da in ds.data_vars.items():
        if all(d in da.dims for d in [tim, lat, lon]): return name
    raise ValueError("Variável SST não localizada.")

def _to_celsius(da):
    units = str(da.attrs.get("units","")).lower()
    sample = float(np.nanmean(da.values[:1].ravel())) if da.size else np.nan
    if ("k" == units) or units.startswith("kelvin") or (np.isfinite(sample) and sample > 100):
        out = da - 273.15
        out.attrs["units"] = "degree_Celsius"
        return out
    return da

def _unify_lon_pm180(da):
    if "lon" in da.coords and float(da["lon"].max()) > 180:
        lon2 = ((da["lon"] + 180) % 360) - 180
        return da.assign_coords(lon=lon2).sortby("lon")
    return da

def _get_cmap():
    try:
        import cmocean
        return cmocean.cm.thermal
    except Exception:
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "cmocean"])
            import cmocean
            return cmocean.cm.thermal
        except Exception:
            return plt.get_cmap("turbo")

def _ensure_cartopy():
    try:
        import cartopy.crs as ccrs  # noqa
        import cartopy.feature as cfeature  # noqa
        return True
    except Exception:
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "cartopy>=0.22", "shapely", "pyproj"])
            import cartopy.crs as ccrs  # noqa
            import cartopy.feature as cfeature  # noqa
            return True
        except Exception:
            return False

def _interp_field(lat, lon, field, factor=UPSAMPLE_FACTOR):
    lat2 = np.linspace(lat.min(), lat.max(), max(len(lat), len(lat)*factor))
    lon2 = np.linspace(lon.min(), lon.max(), max(len(lon), len(lon)*factor))
    fld2 = field.interp(lat=lat2, lon=lon2, method="linear")
    return lat2, lon2, fld2.values

def _nice_levels(vmin, vmax, step):
    a = np.floor(vmin / step) * step
    b = np.ceil (vmax / step) * step
    return np.arange(a, b + 0.5*step, step)

def _grid_shape(n):
    # layout quase quadrado
    for rows in range(1, 7):
        cols = int(np.ceil(n / rows))
        if rows*cols >= n and cols-rows <= 2:
            return rows, cols
    s = int(np.ceil(np.sqrt(n)))
    return s, s

# ===========================
# Dados e plotagem
# ===========================
def load_event_window(product, region_name, event):
    months = _event_months(event)
    das = []
    for y, m in months:
        if product.upper() == "ERA5":
            p = _era5_path(region_name, y, m); assert _exist_ok(p), f"Arquivo ausente: {p}"
            ds = xr.open_dataset(p, engine="netcdf4")
            lat, lon, tim = _discover_coords(ds)
            var = _find_sst_var(ds)
            da  = _to_celsius(ds[var]).rename({lat:"lat", lon:"lon", tim:"time"})
            daD = da.resample(time="1D").mean("time", keep_attrs=True)
            das.append(daD)
            ds.close()
        else:
            p = _oisst_path(region_name, y, m); assert _exist_ok(p), f"Arquivo ausente: {p}"
            ds = xr.open_dataset(p, engine="netcdf4")
            lat, lon, tim = _discover_coords(ds)
            var = _find_sst_var(ds)
            da  = ds[var].rename({lat:"lat", lon:"lon", tim:"time"}).astype("float32")
            das.append(da)
            ds.close()
    da_all = xr.concat(das, dim="time").sortby("time")
    da_all = _unify_lon_pm180(da_all)
    t0 = _as_naive_utc(event["start"]); t1 = _as_naive_utc(event["end"])
    out = da_all.sel(time=slice(t0, t1)).rename("sst")
    # garante latitude crescente
    if float(out.lat[0]) > float(out.lat[-1]):
        out = out.reindex(lat=out.lat[::-1])
    return out

def plot_event_mosaic(product, region_name, event_name, cfg, regs):
    event  = cfg["events"][event_name]
    da_evt = load_event_window(product, region_name, event)  # °C
    dates  = pd.date_range(_as_naive_utc(event["start"]).normalize(),
                           _as_naive_utc(event["end"]).normalize(), freq="D")

    # escala robusta no período
    vmin = float(da_evt.quantile(0.05))
    vmax = float(da_evt.quantile(0.95))
    levels = _nice_levels(vmin, vmax, ISO_DX)
    cmap = _get_cmap()
    source_long = "NOAA OISST v2.1 (AVHRR)" if product.upper()=="OISST" else "ECMWF ERA5 (single-levels)"

    use_cartopy = _ensure_cartopy()
    if use_cartopy:
        import cartopy.crs as ccrs
        import cartopy.feature as cfeature
        proj = ccrs.PlateCarree()

    n = len(dates)
    nrows, ncols = _grid_shape(n)

    # figura com COLUNA extra para o colorbar (não sobrepõe painéis)
    fig_w = max(FIGSIZE_BASE[0], 3.6*ncols + 1.0)
    fig_h = max(FIGSIZE_BASE[1], 3.2*nrows)
    fig = plt.figure(figsize=(fig_w, fig_h))
    gs  = fig.add_gridspec(nrows=nrows, ncols=ncols+1, width_ratios=[1]*ncols + [0.04], wspace=0.08, hspace=0.10)

    axes = []
    for r in range(nrows):
        row_axes = []
        for c in range(ncols):
            if use_cartopy:
                ax = fig.add_subplot(gs[r, c], projection=proj)
            else:
                ax = fig.add_subplot(gs[r, c])
            row_axes.append(ax)
        axes.append(row_axes)
    axes = np.array(axes)

    cax = fig.add_subplot(gs[:, -1])  # eixo exclusivo da barra de cores

    lat = da_evt.lat.values
    lon = da_evt.lon.values
    x_extent = [float(lon.min()), float(lon.max())]
    y_extent = [float(lat.min()), float(lat.max())]

    last_im = None
    for i, d in enumerate(dates):
        r, c = divmod(i, ncols)
        ax = axes[r, c]

        da_day = da_evt.sel(time=d, method="nearest").transpose("lat","lon")
        lat_i, lon_i, fld = _interp_field(da_day.lat.values, da_day.lon.values, da_day)

        if use_cartopy:
            ax.set_extent([*x_extent, *y_extent], crs=proj)
            last_im = ax.imshow(fld, origin="lower",
                                extent=[lon_i.min(), lon_i.max(), lat_i.min(), lat_i.max()],
                                transform=proj, interpolation="bilinear",
                                cmap=cmap, vmin=vmin, vmax=vmax)
            cs = ax.contour(lon_i, lat_i, fld, levels=levels, colors="gray",
                            alpha=0.5, linewidths=0.6, transform=proj)
            if SHOW_ISO_LABELS:
                try: ax.clabel(cs, fmt=lambda x: f"{x:.0f}°C", inline=True, fontsize=7, inline_spacing=2)
                except Exception: pass
            ax.coastlines(resolution="110m", linewidth=0.7)
            ax.add_feature(cfeature.BORDERS, linewidth=0.3)
        else:
            last_im = ax.imshow(fld, origin="lower",
                                extent=[lon_i.min(), lon_i.max(), lat_i.min(), lat_i.max()],
                                interpolation="bilinear", cmap=cmap, vmin=vmin, vmax=vmax, aspect="auto")
            cs = ax.contour(lon_i, lat_i, fld, levels=levels, colors="gray", alpha=0.5, linewidths=0.6)
            if SHOW_ISO_LABELS:
                try: ax.clabel(cs, fmt=lambda x: f"{x:.0f}°C", inline=True, fontsize=7)
                except Exception: pass
            ax.set_xlim(*x_extent); ax.set_ylim(*y_extent)
            ax.set_xlabel("Lon (°)"); ax.set_ylabel("Lat (°)")

        ax.set_title(pd.to_datetime(da_day.time.values).strftime("%Y-%m-%d"), fontsize=10)

    # apaga painéis sobrando
    for j in range(n, nrows*ncols):
        r, c = divmod(j, ncols)
        axes[r, c].axis("off")

    # colorbar dedicado
    cb = fig.colorbar(last_im, cax=cax, orientation="vertical")
    cb.set_label("SST (°C)")

    # título geral + fonte
    t0 = _as_naive_utc(event["start"]).strftime("%Y-%m-%d")
    t1 = _as_naive_utc(event["end"]).strftime("%Y-%m-%d")
    fig.suptitle(f"{product} —  {event_name}  |  {region_name}  |  {t0} → {t1}", y=0.995, fontsize=13)
    fig.text(0.01, 0.01, f"Fonte do dado: {source_long}",
             ha="left", va="bottom", fontsize=9,
             bbox=dict(boxstyle="round,pad=0.25", facecolor="white", alpha=0.6, linewidth=0))

    # salvar
    out_dir = f"{PLOTS_ROOT}/{product}/{region_name}"
    Path(out_dir).mkdir(parents=True, exist_ok=True)
    out_fn  = f"{out_dir}/MOSAICO_{product}_{region_name}_{event_name}_{t0}_{t1}.png"
    plt.savefig(out_fn, dpi=SAVE_DPI, bbox_inches="tight")
    plt.show()
    print("Figura salva:", out_fn)

# ===========================
# Execução
# ===========================
cfg, REGIONS = _open_cfg(CFG_PATH)
print(f"Config: product={PRODUCT} | região={REGION_FOR_DATA} {REGIONS.get(REGION_FOR_DATA,'?')}")
for ev in EVENTS_TO_PLOT:
    if ev not in cfg["events"]:
        print(f"[AVISO] Evento '{ev}' não encontrado no config; ignorado.")
        continue
    plot_event_mosaic(PRODUCT, REGION_FOR_DATA, ev, cfg, REGIONS)


Config: product=OISST | região=SAO_FULL {'north': 0.0, 'west': -70.0, 'south': -60.0, 'east': 20.0}


AssertionError: Arquivo ausente: /content/drive/MyDrive/01_ESTUDOS/04_OCEANOGRAFIA/06_DADOS/OISST/SST/raw/SAO_FULL/2008/oisst_sst_SAO_FULL_200811.nc

In [ ]:
# ===========================
# PASSO 7B — Séries + Anomalias (barras) + Histograma
# Loop em TODOS os eventos; salva PNGs organizados por região.
# Gráfico A melhorado: faixa P10–P90, pontos diários, linhas médias, janela do evento,
# legenda fora (sem sobreposição), datas compactas.
# ===========================
import os, json
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

# -------- parâmetros --------
REGION_FOR_DATA = "SAO_FULL"   # região-alvo
PRE_DAYS, POST_DAYS = 10, 10   # contexto antes/depois
ROLL_N = 3                     # média móvel central (dias)
SAVE_FIG = True

# -------- paths (reaproveita se já existem) --------
try:
    DRIVE_ROOT
except NameError:
    DRIVE_ROOT = "/content/drive/MyDrive/01_ESTUDOS/04_OCEANOGRAFIA"
DIR_DADOS      = f"{DRIVE_ROOT}/06_DADOS"
CFG_PATH       = f"{DRIVE_ROOT}/05_CODIGOS/mhw_config.json"
ERA5_RAW_ROOT  = f"{DIR_DADOS}/ERA5/SST/raw"
OISST_RAW_ROOT = f"{DIR_DADOS}/OISST/SST/raw"
OUT_DIR_ROOT   = f"{DRIVE_ROOT}/07_PLOTAGENS/SERIES"
Path(OUT_DIR_ROOT).mkdir(parents=True, exist_ok=True)

# -------- helpers (definidos só se faltarem) --------
def _as_naive_utc(ts): return pd.to_datetime(ts, utc=True).tz_localize(None)
def _exist_ok(p): return os.path.exists(p) and os.path.getsize(p) > 0
def _era5_path(region, y, m):  return f"{ERA5_RAW_ROOT}/{region}/{y}/era5_sst_{region}_{y}{m:02d}.nc"
def _oisst_path(region, y, m): return f"{OISST_RAW_ROOT}/{region}/{y}/oisst_sst_{region}_{y}{m:02d}.nc"

def _open_cfg(path):
    with open(path, "r") as f:
        cfg = json.load(f)
    regs = {}
    for k, v in cfg["regions"].items():
        if "area" in v:
            N, W, S, E = v["area"]
            regs[k] = dict(north=float(N), west=float(W), south=float(S), east=float(E))
        else:
            vv = {kk.lower(): vv for kk, vv in v.items()}
            regs[k] = dict(north=float(vv["north"]), west=float(vv["west"]),
                           south=float(vv["south"]), east=float(vv["east"]))
    return cfg, regs

def _discover_coords(ds):
    def pick(cands):
        for c in cands:
            if c in ds.coords or c in ds.dims: return c
        for name, da in ds.variables.items():
            if getattr(da, "ndim", 0) == 1:
                std = str(da.attrs.get("standard_name","")).lower()
                if "latitude" in std:  return name
                if "longitude" in std: return name
                if "time" in std:      return name
        return None
    return pick(["latitude","lat","y"]), pick(["longitude","lon","x"]), pick(["time","valid_time","t"])

def _find_sst_var(ds):
    for k in ["sea_surface_temperature","sst","SST","sstk","SSTK","analysis_sst"]:
        if k in ds.data_vars or k in ds.variables: return k
    lat, lon, tim = _discover_coords(ds)
    for name, da in ds.data_vars.items():
        if all(d in da.dims for d in [tim, lat, lon]): return name
    raise ValueError("Variável SST não localizada.")

def _to_celsius(da):
    units = str(da.attrs.get("units","")).lower()
    sample = float(np.nanmean(da.values[:1].ravel())) if da.size else np.nan
    if ("k" == units) or units.startswith("kelvin") or (np.isfinite(sample) and sample > 100):
        out = da - 273.15
        out.attrs["units"] = "degree_Celsius"
        return out
    return da

def _unify_lon_pm180(da):
    if "lon" in da.coords and float(da["lon"].max()) > 180:
        lon2 = ((da["lon"] + 180) % 360) - 180
        return da.assign_coords(lon=lon2).sortby("lon")
    return da

def _event_months(e):
    t0 = pd.to_datetime(e["start"]); t1 = pd.to_datetime(e["end"])
    per = pd.period_range(t0.to_period("M"), t1.to_period("M"), freq="M")
    return [(p.year, p.month) for p in per]

def load_event_window(product, region_name, event):
    months = _event_months(event)
    out = []
    for y, m in months:
        if product.upper() == "ERA5":
            p = _era5_path(region_name, y, m); assert _exist_ok(p), f"Arquivo ausente: {p}"
            ds = xr.open_dataset(p, engine="netcdf4")
            lat, lon, tim = _discover_coords(ds); var = _find_sst_var(ds)
            da = _to_celsius(ds[var]).rename({lat:"lat", lon:"lon", tim:"time"})
            out.append(da.resample(time="1D").mean("time", keep_attrs=True))
            ds.close()
        else:
            p = _oisst_path(region_name, y, m); assert _exist_ok(p), f"Arquivo ausente: {p}"
            ds = xr.open_dataset(p, engine="netcdf4")
            lat, lon, tim = _discover_coords(ds); var = _find_sst_var(ds)
            out.append(ds[var].rename({lat:"lat", lon:"lon", tim:"time"}).astype("float32"))
            ds.close()
    da = xr.concat(out, dim="time").sortby("time")
    return _unify_lon_pm180(da).rename("sst")

def normalize_to_daily_index(da):
    t = pd.to_datetime(da.time.values)
    try: t = t.tz_localize(None)
    except Exception: pass
    return da.assign_coords(time=("time", t)).resample(time="1D").mean("time", keep_attrs=True)

def reindex_daily(da, t0, t1):
    rng = pd.date_range(t0, t1, freq="D")
    return normalize_to_daily_index(da).reindex(time=rng)

def area_stats(da):
    w = np.cos(np.deg2rad(da["lat"])).clip(min=0)
    w = w / w.mean()
    m   = (da * w).mean(dim=("lat","lon"), skipna=True)
    p10 = da.quantile(0.10, dim=("lat","lon"), skipna=True)
    p90 = da.quantile(0.90, dim=("lat","lon"), skipna=True)
    return m, p10, p90

def roll_center(x, N):
    n = x.sizes.get("time", x.shape[0])
    w = max(1, min(N, int(n)))
    return x.rolling(time=w, center=True, min_periods=1).mean()

# -------- carrega config --------
cfg, REGIONS = _open_cfg(CFG_PATH)
events_dict = cfg["events"]
events_sorted = sorted(events_dict.items(), key=lambda kv: pd.to_datetime(kv[1]["start"]))

# -------- loop --------
out_dir_region = f"{OUT_DIR_ROOT}/{REGION_FOR_DATA}"
Path(out_dir_region).mkdir(parents=True, exist_ok=True)

saved = []
for EVENT_NAME, event in events_sorted:
    try:
        # janela
        t0_evt = _as_naive_utc(event["start"])
        t1_evt = _as_naive_utc(event["end"])
        t_peak = _as_naive_utc(event.get("peak", t0_evt)).normalize()
        t0 = (t0_evt - pd.Timedelta(days=PRE_DAYS)).normalize()
        t1 = (t1_evt + pd.Timedelta(days=POST_DAYS)).normalize()

        # dados (diários e alinhados)
        da_oi = reindex_daily(load_event_window("OISST", REGION_FOR_DATA, event), t0, t1)
        da_e5 = reindex_daily(load_event_window("ERA5",  REGION_FOR_DATA, event), t0, t1)

        # estatísticas
        m_oi, q10_oi, q90_oi = area_stats(da_oi)
        m_e5, q10_e5, q90_e5 = area_stats(da_e5)
        m_oi_s, m_e5_s = roll_center(m_oi, ROLL_N), roll_center(m_e5, ROLL_N)

        # baseline (PRE_DAYS)
        base_oi = float(m_oi.sel(time=slice(t0, t0_evt - pd.Timedelta(days=1))).mean(skipna=True))
        base_e5 = float(m_e5.sel(time=slice(t0, t0_evt - pd.Timedelta(days=1))).mean(skipna=True))
        anom_oi, anom_e5 = m_oi - base_oi, m_e5 - base_e5

        # diferenças comuns
        common_all  = np.intersect1d(m_oi.time.values, m_e5.time.values)
        common_bars = np.intersect1d(anom_oi.time.values, anom_e5.time.values)
        diffs = roll_center((m_e5.sel(time=common_all) - m_oi.sel(time=common_all)), ROLL_N)

        # histograma (evento)
        evt_common = np.intersect1d(
            m_oi.sel(time=slice(t0_evt, t1_evt)).time.values,
            m_e5.sel(time=slice(t0_evt, t1_evt)).time.values
        )
        h_oi = np.asarray(m_oi.sel(time=evt_common).values)
        h_e5 = np.asarray(m_e5.sel(time=evt_common).values)
        rmse = float(np.sqrt(np.nanmean((h_e5 - h_oi)**2))) if h_oi.size and h_e5.size else np.nan
        bias = float(np.nanmean(h_e5 - h_oi)) if h_oi.size and h_e5.size else np.nan

        # ---------- FIGURA ----------
        plt.close("all")
        fig = plt.figure(figsize=(14.8, 9.4))
        gs = fig.add_gridspec(2, 2, height_ratios=[2.2, 1.25],
                              width_ratios=[1.35, 1.0], hspace=0.38, wspace=0.34)

        # (A) Série temporal melhorada
        ax = fig.add_subplot(gs[0, :])

        # faixas P10–P90 (como ribbons) — cores discretas, alpha leve
        ax.fill_between(q10_oi["time"].values, q10_oi.values, q90_oi.values,
                        color="#f39c12", alpha=0.12, zorder=1)
        ax.fill_between(q10_e5["time"].values, q10_e5.values, q90_e5.values,
                        color="#3498db", alpha=0.10, zorder=1)

        # pontos diários (pequenos) + linhas suavizadas
        ax.plot(m_oi["time"].values, m_oi.values, "o", ms=2.8, color="#e67e22",
                alpha=0.75, zorder=3)
        ax.plot(m_e5["time"].values, m_e5.values, "o", ms=2.8, color="#1f77b4",
                alpha=0.75, zorder=3)
        ax.plot(m_oi_s["time"].values, m_oi_s.values, lw=2.2, color="#e67e22", zorder=4)
        ax.plot(m_e5_s["time"].values, m_e5_s.values, lw=2.2, color="#1f77b4", zorder=4)

        # janela do evento + pico (faixa vertical em axvspan)
        ax.axvspan(t0_evt, t1_evt, color="k", alpha=0.06, zorder=0)
        ax.axvline(t_peak, color="k", ls="--", lw=1.0, zorder=5)

        # eixos e estilo
        ax.set_ylabel("SST (°C)")
        ax.set_title(f"Série temporal — {EVENT_NAME} | {REGION_FOR_DATA}")
        ax.grid(True, lw=0.3, alpha=0.35)
        loc = mdates.AutoDateLocator()
        ax.xaxis.set_major_locator(loc)
        ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(loc))  # rótulos de data compactos

        # legenda fora (topo, central) com handles personalizados (nada sobrepõe)
        handles = [
            Patch(facecolor="#f39c12", alpha=0.12, label="OISST P10–P90"),
            Patch(facecolor="#3498db", alpha=0.10, label="ERA5 P10–P90"),
            Line2D([], [], marker="o", ls="", color="#e67e22", markersize=5, label="OISST diária"),
            Line2D([], [], marker="o", ls="", color="#1f77b4", markersize=5, label="ERA5 diária"),
            Line2D([], [], color="#e67e22", lw=2.2, label=f"OISST média {ROLL_N}d"),
            Line2D([], [], color="#1f77b4", lw=2.2, label=f"ERA5 média {ROLL_N}d"),
            Line2D([], [], color="k", ls="--", lw=1.0, label="Pico"),
        ]
        legA = ax.legend(handles=handles, ncols=3, loc="upper center",
                         bbox_to_anchor=(0.5, 1.16), frameon=True, framealpha=0.96,
                         facecolor="white", borderpad=0.35)
        legA.set_zorder(20)

        # (B) Barras (anomalias) + linha de diferença
        axb = fig.add_subplot(gs[1, 0], sharex=ax)
        anom_oi_b = anom_oi.sel(time=common_bars)
        anom_e5_b = anom_e5.sel(time=common_bars)
        xnum = mdates.date2num(pd.to_datetime(anom_oi_b["time"].values))
        width = 0.35  # dias
        axb.bar(xnum - width/2, anom_oi_b.values, width=width,
                color="#e67e22", alpha=0.75, label=f"OISST anom (base {PRE_DAYS}d)", zorder=2)
        axb.bar(xnum + width/2, anom_e5_b.values, width=width,
                color="#1f77b4", alpha=0.75, label=f"ERA5 anom (base {PRE_DAYS}d)", zorder=2)
        axb.axhline(0, color="k", lw=0.8, zorder=1)
        axb.set_ylabel("Anomalia (°C)")
        axb.grid(True, lw=0.3, alpha=0.35, zorder=0)
        for lab in axb.get_xticklabels():
            lab.set_rotation(30); lab.set_ha("right")

        axb2 = axb.twinx()
        if diffs.size:
            axb2.plot(diffs["time"].values, diffs.values, "-", color="0.25",
                      lw=1.8, label=f"ERA5 − OISST (média {ROLL_N}d)", zorder=3)
        # legendas fora (acima das barras)
        legB = axb.legend(loc="upper left", bbox_to_anchor=(0.01, 1.02),
                          frameon=True, framealpha=0.96, facecolor="white")
        legB.set_zorder(20)
        if diffs.size:
            legB2 = axb2.legend(loc="upper right", bbox_to_anchor=(0.99, 1.02),
                                frameon=True, framealpha=0.96, facecolor="white")
            legB2.set_zorder(20)

        # (C) Histograma do evento — legenda fora à direita (nada sobrepõe)
        axh = fig.add_subplot(gs[1, 1])
        if h_oi.size and h_e5.size:
            f_oi = h_oi[np.isfinite(h_oi)]; f_e5 = h_e5[np.isfinite(h_e5)]
            if f_oi.size and f_e5.size:
                allv = np.concatenate([f_oi, f_e5])
                span = float(np.nanmax(allv) - np.nanmin(allv))
                nb   = int(np.clip(np.ceil(span / 0.1), 10, 25))
                bins = np.linspace(np.nanmin(allv), np.nanmax(allv), nb)
                axh.hist(f_oi, bins=bins, alpha=0.65, label="OISST (evento)",
                         color="#e67e22", edgecolor="none", density=True, zorder=2)
                axh.hist(f_e5, bins=bins, alpha=0.65, label="ERA5 (evento)",
                         color="#1f77b4", edgecolor="none", density=True, zorder=2)
                axh.axvline(np.nanmedian(f_oi), color="#e67e22", lw=1.6, ls="--", zorder=3)
                axh.axvline(np.nanmedian(f_e5), color="#1f77b4", lw=1.6, ls="--", zorder=3)
                # legenda FORA (direita)
                legh = axh.legend(loc="center left", bbox_to_anchor=(1.02, 0.5),
                                  frameon=True, framealpha=0.96, facecolor="white")
                legh.set_zorder(20)
                # métricas acima, fora do eixo
                axh.annotate(f"Viés (ERA5−OISST): {bias:+.2f} °C   RMSE: {rmse:.2f} °C",
                             xy=(1.0, 1.08), xycoords="axes fraction",
                             ha="right", va="bottom", fontsize=9,
                             bbox=dict(boxstyle="round,pad=0.22", facecolor="white",
                                       alpha=0.96, linewidth=0), zorder=20)
        axh.set_xlabel("SST (°C)")
        axh.set_ylabel("Densidade")
        axh.grid(True, lw=0.3, alpha=0.35, zorder=0)
        axh.margins(x=0.05)

        # rodapé
        src = "Fonte do dado: NOAA OISST v2.1 (AVHRR)  +  ECMWF ERA5 (single-levels)"
        fig.text(0.01, 0.012, src, fontsize=9)

        # layout com sobras para legendas/anotações fora dos eixos
        plt.tight_layout(rect=(0, 0.05, 1, 0.97))

        # salvar
        if SAVE_FIG:
            fname = f"{out_dir_region}/SERIE_{EVENT_NAME}_{REGION_FOR_DATA}.png"
            plt.savefig(fname, dpi=200)
            saved.append(fname)
            print(f"[OK] {EVENT_NAME}: figura salva → {fname}")
        plt.show()

    except Exception as e:
        print(f"[ERRO] {EVENT_NAME}: {e}")

print("\nConcluído. Arquivos gerados:")
for f in saved:
    print(" -", f)


In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

def to_celsius(da):
    return da - 273.15 if da.attrs.get("units","").lower() in ["k","kelvin"] else da

def subset_bbox(da, bbox):
    n,w,s,e = bbox
    lon = da.lon.copy()
    if lon.max()>180: lon = ((lon+180)%360)-180
    da = da.assign_coords(lon=lon).sortby("lon")
    if w<=e:
        da = da.sel(lat=slice(n,s), lon=slice(w,e))
    else:
        da1 = da.sel(lat=slice(n,s), lon=slice(w,da.lon.max()))
        da2 = da.sel(lat=slice(n,s), lon=slice(da.lon.min(),e))
        da = xr.concat([da1,da2], dim="lon")
    return da

def dayofyear_366(t):
    doy = xr.DataArray(t.dt.dayofyear, coords={"time":t}, dims="time")
    feb29 = (t.dt.month==2)&(t.dt.day==29)
    doy = xr.where(feb29, 366, doy)
    return doy

def climatology_and_threshold(da, baseline_start, baseline_end, window_days=31, perc=90, smooth_days=31):
    base = da.sel(time=slice(np.datetime64(baseline_start), np.datetime64(baseline_end)))
    t = base.time
    doy = dayofyear_366(t)
    base = base.groupby(doy).apply(lambda x: x)
    half = window_days//2
    vals = []
    for d in range(1,367):
        idx = [(d+i-1)%366+1 for i in range(-half,half+1)]
        sub = xr.concat([base.sel(doy=k) for k in idx], dim="time")
        vals.append(xr.concat([sub.mean("time"), sub.reduce(np.nanpercentile, q=perc, dim="time")], dim="metric").assign_coords(metric=["mean","p"]))
    clim_stack = xr.concat(vals, dim="doy")
    clim = clim_stack.sel(metric="mean")
    thr = clim_stack.sel(metric="p")
    if smooth_days and smooth_days>1:
        clim = clim.rolling(doy=smooth_days, center=True, min_periods=1).mean()
        thr  = thr.rolling(doy=smooth_days, center=True, min_periods=1).mean()
    return clim, thr

def expand_daily_to_time(daily_field, time):
    doy = dayofyear_366(time)
    return daily_field.sel(doy=doy.values)

def detect_events(exceed_bool, min_duration=5):
    idx = exceed_bool.values.astype(np.int8)
    starts = []
    ends = []
    i = 0
    n = idx.size
    while i<n:
        if idx[i]==1:
            j=i
            while j<n and idx[j]==1:
                j+=1
            if j-i>=min_duration:
                starts.append(i)
                ends.append(j-1)
            i=j
        else:
            i+=1
    return np.array(starts), np.array(ends)

def mhw_metrics(ts, clim_ts, thr_ts, min_duration=5):
    exceed = ts>thr_ts
    starts, ends = detect_events(exceed, min_duration=min_duration)
    if starts.size==0:
        daily = xr.Dataset({"exceed":exceed, "intensity":(ts-thr_ts).where(exceed,0.0), "category":xr.zeros_like(ts)+0})
        return pd.DataFrame(columns=["start","end","duration","peak_date","max_intensity","mean_intensity","cum_intensity","max_category"]), daily
    delta = (thr_ts - clim_ts).where((thr_ts - clim_ts)>0, np.nan)
    inten = (ts - thr_ts).where(exceed, 0.0)
    ratio = (inten/delta).where(exceed)
    cat = xr.where(ratio<1,1,
          xr.where(ratio<2,2,
          xr.where(ratio<3,3,4))).where(exceed,0)
    rows = []
    for s,e in zip(starts, ends):
        seg = inten.isel(time=slice(s,e+1))
        seg_cat = cat.isel(time=slice(s,e+1))
        peak_idx = int(seg.argmax("time"))
        rows.append({
            "start": ts.time.values[s],
            "end": ts.time.values[e],
            "duration": e-s+1,
            "peak_date": ts.time.values[s+peak_idx],
            "max_intensity": float(seg.max().values),
            "mean_intensity": float(seg.mean().values),
            "cum_intensity": float(seg.sum().values),
            "max_category": int(seg_cat.max().values),
        })
    daily = xr.Dataset({"exceed":exceed, "intensity":inten, "category":cat})
    return pd.DataFrame(rows), daily

def regional_series(da):
    w = np.cos(np.deg2rad(da.lat))
    w = w/w.mean()
    return (da.weighted(w).mean(dim=("lat","lon"))).rename("sst")

def fraction_area_exceed(da_bool):
    w = np.cos(np.deg2rad(da_bool.lat))
    w2 = w/ w.sum()
    ones = xr.ones_like(da_bool.isel(time=0, drop=True))
    area = (ones*0+1.0).weighted(w2).sum(("lat","lon"))
    exc = da_bool.where(da_bool,0.0).weighted(w2).sum(("lat","lon"))
    return (exc/area).rename("frac_exceed")

def run_mhw_pipeline(ds, varname, bbox, baseline=("1982-01-01","2011-12-31"), window_days=31, perc=90, smooth_days=31, min_duration=5):
    da = to_celsius(ds[varname]).transpose("time","lat","lon")
    da = subset_bbox(da, bbox)
    reg_ts = regional_series(da)
    clim_daily, thr_daily = climatology_and_threshold(reg_ts, baseline[0], baseline[1], window_days, perc, smooth_days)
    clim_ts = expand_daily_to_time(clim_daily, reg_ts.time)
    thr_ts  = expand_daily_to_time(thr_daily,  reg_ts.time)
    events_df, daily_reg = mhw_metrics(reg_ts, clim_ts, thr_ts, min_duration=min_duration)
    thr_grid = expand_daily_to_time(thr_daily, da.time)
    exc_grid = da>thr_grid
    frac = fraction_area_exceed(exc_grid)
    return {"events":events_df, "daily":{"ts":reg_ts, "clim":clim_ts, "thr":thr_ts, "intensity":daily_reg["intensity"], "category":daily_reg["category"], "frac_exceed":frac}}

def plot_daily_summary(out, title_prefix, savepath=None):
    t = out["daily"]["ts"].time
    plt.figure(figsize=(10,4.5))
    plt.plot(t, out["daily"]["ts"].values, label="SST")
    plt.plot(t, out["daily"]["clim"].values, label="Clim")
    plt.plot(t, out["daily"]["thr"].values, label="P90")
    plt.fill_between(t, out["daily"]["thr"].values, out["daily"]["ts"].values, where=out["daily"]["ts"].values>out["daily"]["thr"].values, alpha=0.25, label="Excesso")
    plt.title(f"{title_prefix} — Série diária")
    plt.xlabel("Data"); plt.ylabel("°C"); plt.legend(); plt.tight_layout()
    if savepath: plt.savefig(savepath, dpi=150, bbox_inches="tight")
    plt.show()
    plt.figure(figsize=(10,3.5))
    plt.plot(t, out["daily"]["intensity"].values)
    plt.title(f"{title_prefix} — Intensidade (°C acima do limiar)")
    plt.xlabel("Data"); plt.ylabel("°C"); plt.tight_layout()
    plt.show()
    plt.figure(figsize=(10,3.5))
    plt.plot(t, out["daily"]["frac_exceed"].values)
    plt.title(f"{title_prefix} — Fração de área acima do limiar")
    plt.xlabel("Data"); plt.ylabel("fração"); plt.tight_layout()
    plt.show()

# Exemplo de uso
# ds = xr.open_dataset("/path/para/oisst_diario.nc", chunks={"time":60})  # var tipicamente "sst"
# bbox = (0.0, -70.0, -60.0, 20.0)
# out = run_mhw_pipeline(ds, varname="sst", bbox=bbox, baseline=("1982-01-01","2011-12-31"), window_days=31, perc=90, smooth_days=31, min_duration=5)
# out["events"].to_csv("/path/para/SAO_FULL_MHW_eventos.csv", index=False)
# plot_daily_summary(out, "SAO_FULL")
